# FeFET MLC 多值 (WGFMU 脉冲) · 引擎门 + inline 看图 (50)

> 2026-06-11 建。**薄 notebook**：测量逻辑全在 `fefetlab.protocols.wgfmu_fefet` 的
> `run_stage_*`(金标准守着、停门/manifest 都在),本本只管 **配置 → 跑 → 画图**。
> 走 `ProtocolEngine.run` 统一门,一段一确认 / 停门 / 归档全在引擎里 ——
> **notebook 与命令行同等安全**,不再手搓波形。
>
> **本本聚焦 ② MLC 多值**(PPT 第6页②)：S0 抬针 smoke → S1 器件 baseline → MLC 扫编程幅值。
> (DC Id-Vg 在 `51_dc_idvg.ipynb`;别的脉冲实验要测了再加 cell。)

## 用法
1. 在**测试机** `D:\test\B1500` 根目录起 `jupyter lab`,打开本本。
2. 改【配置】cell：器件身份 / 接线 / 安全电压。**先 `LIVE=False` dry 跑通全程**(无硬件,验证管线),确认无误再 `LIVE=True` 上真机。
3. 自上而下一段段跑。**live 时一次只跑一个 run cell**,看完 Id–Vg / |Ig| 图再决定下一步。
4. 数据自动落 `runs/<device>/<geometry>_<serial>/{live,dry}/<时间戳>_<stage>/`(CSV+manifest+summary)。

## 安全
- |Ig| 超停门(各段已设)→ **先落盘已采数据再停**,不进下一炮/段。
- 看 |Ig| 图：读电压下 |Ig| 异常抬升 = 栅漏,停手查接线/降压。
- 脆弱器件先把 `WRITE_V` 设小(如 3.0)、`*_reps` 设 1,跑通再加力度。

In [ ]:
# ===================== 【配置】改这里 =====================
DEVICE_ID   = "qzb_sy2"          # 批次/自命名 → 顶层归集文件夹
GEOMETRY    = "L300W300"         # 几何 沟长×沟宽
SERIAL      = "1"                # 序号 / die 号
DEVICE_TYPE = "FeFET"            # 类型(进 manifest)
OPERATOR    = "yhzang"

GATE_CH, DRAIN_CH = 202, 201     # 接线：Gate=WGFMU202(经RSU与SMU2共栅) / Drain=WGFMU201；你这颗若不同改这里

LIVE    = False                  # ← 先 False(dry 审计,无硬件)跑通；确认无误再 True 上真机
VD_READ = None                   # S0/S1 baseline 读 Vd；None=0.05V(MLC 的读 Vd 在下方【MLC 参数】单独设)
# =========================================================
print(f"目标器件: {DEVICE_ID} / {GEOMETRY} #{SERIAL} ({DEVICE_TYPE})   LIVE={LIVE}")

In [ ]:
import sys
from pathlib import Path

# 定位 repo 根(含 src/fefetlab),无论从 notebooks/ 还是根目录启动都能找到
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "fefetlab").exists())
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt

from fefetlab.engine import ProtocolEngine, REGISTRY
from fefetlab.protocols.wgfmu_fefet import parse_args, make_backend, configure_channel_map


def _argv(stage, **ov):
    a = ["--stage", stage, "--device-id", DEVICE_ID, "--geometry", GEOMETRY,
         "--serial", SERIAL, "--device-type", DEVICE_TYPE, "--operator", OPERATOR,
         "--gate-ch", str(GATE_CH), "--drain-ch", str(DRAIN_CH)]
    if LIVE:
        a.append("--live")
    if VD_READ is not None:
        a += ["--vd-read", str(VD_READ)]
    for k, v in ov.items():                       # 任意 stage 参数：e1_reps=1 → --e1-reps 1
        a += ["--" + k.replace("_", "-"), str(v)]
    return a


def run_stage(stage, **ov):
    # 走引擎门跑一段。live 自动 confirm=stage(一段一确认仍由引擎强制)。返回 StageSummary。
    args = parse_args(_argv(stage, **ov))
    configure_channel_map(args)                   # 设 GATE_CH/DRAIN_CH 全局(make_backend 要读)
    backend, addr = make_backend(LIVE)
    try:
        s = ProtocolEngine().run(stage, vars(args), backend=backend,
                                 confirm=(stage if LIVE else ""))
    finally:
        try:
            backend.close_session()
        except Exception:
            pass
    print(f"[{stage}] {s.report_code}  rows={s.rows}  "
          f"|Id|max={s.max_abs_id_a:.3e}A  |Ig|max={s.max_abs_ig_a:.3e}A")
    print(f"CSV: {s.out_csv}")
    return s


def load(s):
    return pd.read_csv(s.out_csv)


def plot_idvg(s, title=None):
    # |Id|-Vg(按 state 分色) + |Ig| 栅漏监看。适用 S0/S1/E5 等读类。
    df = load(s)
    fig, ax = plt.subplots(figsize=(6, 4))
    for st, g in df.groupby("state_target"):
        g = g.sort_values("Vg_read_V")
        ax.errorbar(g["Vg_read_V"], g["Id_mean_A"].abs(), yerr=g["Id_std_A"],
                    marker="o", capsize=3, label=f"state={st}")
    ax.set_yscale("log")
    ax.set_xlabel("Vg_read (V)")
    ax.set_ylabel("|Id| (A)")
    ax.set_title(title or f"{s.stage}  Id-Vg")
    ax.legend()
    ax.grid(True, which="both", alpha=.3)
    plt.show()

    fig2, ax2 = plt.subplots(figsize=(6, 2.4))
    ax2.plot(df["Vg_read_V"], df["Ig_mean_A"].abs(), "r.-")
    ax2.set_yscale("log")
    ax2.set_xlabel("Vg_read (V)")
    ax2.set_ylabel("|Ig| (A)")
    ax2.set_title("栅漏监看 |Ig|")
    ax2.grid(alpha=.3)
    plt.show()


def plot_mlc(s, title=None):
    # 多值:|Id|-vs-编程幅值(幅值从 dose_mode 解析,同幅值取均值)
    df = load(s)
    amp = df["dose_mode"].str.extract(r"prog_amp=([+-]?[0-9.eE]+)").astype(float)[0]
    g = df.assign(prog_amp=amp).groupby("prog_amp")["Id_mean_A"].mean().abs()
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(g.index, g.values, "o-")
    ax.set_yscale("log")
    ax.set_xlabel("编程幅值 (V)")
    ax.set_ylabel("|Id| (A)")
    ax.set_title(title or f"{s.stage}  Id-vs-编程幅值(多值)")
    ax.grid(True, which="both", alpha=.3)
    plt.show()


print("引擎就绪。可跑协议:", ", ".join(REGISTRY))

## S0 · 空夹具 / 抬针 smoke（先抬针，**不接器件**）
确认链路/通道正常、无异常漏电。|Ig| 应在 baseline 量级（探针抬起时电缆寄生 ~1µA 正常）。

In [ ]:
s0 = run_stage("S0", s0_reps=3)
plot_idvg(s0, title="S0 空夹具 smoke")

## S1 · 器件只读 baseline（探针**落到器件**后再跑）
读 Id–Vg baseline，确认器件活着、电流量级合理，再决定进 MLC。
自定义读 Vg 点：`run_stage("S1", s1_vg="-1.0,-0.5,0,0.5", s1_reps=5)`

In [ ]:
s1 = run_stage("S1")
plot_idvg(s1, title="S1 器件 baseline Id-Vg")

## ② 多值 MLC（PPT 第6页②）：擦除 → 编程@幅值 → 单点读
每个正编程幅值前先固定擦除 reset 到同起点,再编程,单点读 Id。扫幅值出 **Id-vs-编程幅值** 多值曲线。
下面【MLC 参数】cell 就是你要改的地方(默认已按你 PPT②③ 填好,想改直接改值)。

In [ ]:
# =============== 【MLC 参数】PPT 第6页②③,改这里 ===============
MLC_AMPS    = "1.5,2.0,2.5,3.0,3.5,3.8"  # 编程正幅值 V(逗号分隔,扫这些出多值)
MLC_V_ERASE = 4.0          # 每发编程前擦除幅值(绝对值,实际打 -4V),把器件 reset 到同起点
MLC_WIDTH_S = 50e-6        # 擦/编程脉宽 s（50µs）
MLC_READ_VG = 0.5          # 编程后读取 Vg（V）
MLC_READ_VD = 0.1          # 编程后读取 Vd（V）
MLC_REPS    = 1            # 每个幅值重复几次（脆弱先 1）
# =============================================================
print(f"MLC: 幅值[{MLC_AMPS}]V  擦除 -{MLC_V_ERASE}V  脉宽 {MLC_WIDTH_S}s  "
      f"读 Vg{MLC_READ_VG}/Vd{MLC_READ_VD}  reps {MLC_REPS}")

In [ ]:
mlc = run_stage("MLC", mlc_amps=MLC_AMPS, mlc_v_erase=MLC_V_ERASE, mlc_width_s=MLC_WIDTH_S,
                mlc_read_vg=MLC_READ_VG, mlc_vd_read=MLC_READ_VD, mlc_reps=MLC_REPS)
plot_mlc(mlc)